**Cell #01**

# RAG11 Nutrition — Stage 4: Parent-Chunk Expansion ("Small-to-Big") Examples

Adds **parent-chunk expansion** on top of the Stage 2 retrieval + generation
pipeline: after retrieval (plain, hybrid, and/or reranked) has picked its
best-matching *child* chunks, each winning child chunk's text is swapped for
its *parent* chunk's text — the full section it was carved out of — before
Claude ever sees it. Demonstrated with 3 nutrition-specific examples chosen
to show where a narrow child chunk falls short and expansion fixes it.

## The problem, in one picture

Child chunks are deliberately small (~300–500 tokens) so they match a
question **precisely** — but a small chunk sometimes doesn't carry enough
surrounding context to answer the question **fully**. Say a nutrition RAG
gets asked *"How does soluble fiber's effect on LDL cholesterol differ from
insoluble fiber's effect?"*:

- The winning child chunk (after rerank, hybrid, or plain vector search)
  might be just two sentences: *"Soluble fiber binds bile acids in the gut,
  which forces the liver to pull more LDL cholesterol from the blood to make
  more bile acids."*
- That's a **great match** for the question — but it says **nothing about
  insoluble fiber**. The comparison Claude needs is incomplete, even though
  retrieval "worked."
- The data already has the fix on hand: every child chunk's
  `rowParentGUID` points at a bigger parent chunk from Stage 1.1 — the full
  section it was carved out of, which usually discusses both sides of a
  comparison like this together.
- **The fix:** after finding the best small chunk(s), swap them for their
  parent chunk(s) before sending anything to Claude. The small chunk finds
  the needle; the parent chunk gives the whole haystack around the needle,
  so the answer isn't cut off mid-thought.

Think of it like: child chunk = *"here's the exact sentence that matches,"*
parent chunk = *"here's the whole section that sentence lives in."*

## Dedup — the other half of the mechanism

A "compare X and Y" question routinely retrieves *several* winning child
chunks from the very same section (one about X, one about Y). Expanding
each independently would send Claude the same parent section two or three
times over. `expand_to_parent_chunks()` fetches and sends each **distinct**
parent exactly once, recording every child chunk that matched it in
`matched_children`.

## Prerequisites

**No SQL migration needed** — unlike hybrid search, parent-chunk expansion
needs **zero** schema change: `rag11_chunks_parent_table` and the child
table's `rowParentGUID` foreign key already exist
(`sql/create_sql_tables.sql`). It's a pure query-time Python step on top of
whatever `stage2_ask_examples1/2/3.ipynb` already retrieve.

This notebook gets its retrieval/generation logic from `./reusable_code/`
— the same package every other `stage2_ask_examples*.ipynb` imports from —
so all four notebooks share one implementation of `ask_question`
(`reusable_code/generation.py`), `retrieve_chunks`
(`reusable_code/retrieval.py`), `hybrid_search`
(`reusable_code/hybrid_search.py`), and `expand_to_parent_chunks`
(`reusable_code/parent_chunk_expansion.py`).
See `documentation/HOW_IT_WORKS_Parent_Chunk_Expansion.html` for the full
write-up and `reusable_code/README.md` for the one-paragraph summary.


In [ ]:
from reusable_code import (
    init_clients,
    ask_question,
    retrieve_chunks,
    rerank_chunks,
    hybrid_search,
    expand_to_parent_chunks,
    build_expanded_context_block,
    page_numbers_for_expanded_chunk,
    NUM_CONTEXT_CHUNKS,
    DEFAULT_MAX_PARENT_CHARS,
    EMBEDDING_MODEL,
    RERANK_MODEL,
    GENERATION_MODEL,
)

clients = init_clients()
print("Clients ready.")
print("Embedding model:", EMBEDDING_MODEL, "| Rerank model:", RERANK_MODEL,
      "| Generation model:", GENERATION_MODEL,
      "| Default max parent chars:", DEFAULT_MAX_PARENT_CHARS)


**Cell #03**

## A helper to see "before expansion" vs. "after expansion" side by side

`show_expansion_comparison()` reranks a wide candidate pool down to the top
matches (the same "cast a wide net, then pick precisely" pattern
`stage2_ask_examples2_rerank.ipynb` uses), then runs
`expand_to_parent_chunks()` on the result — printed together so the effect
of expansion is visible rather than assumed: how many distinct parents the
winning chunks collapsed into, how much longer the expanded text is, and
how the page-citation range changes once a parent's own
`start_page`/`end_page` replace a single child chunk's narrower range.


In [ ]:
def show_expansion_comparison(question: str, pool: int = 15, top_n: int = NUM_CONTEXT_CHUNKS):
    """Rerank down to `top_n` child chunks for `question`, then run
    expand_to_parent_chunks() on them, printing both stages side by side.
    Returns (children, expanded) so the caller can inspect or reuse either."""
    print(f"Q: {question}\n")

    candidates = retrieve_chunks(question, match_count=pool)
    children = rerank_chunks(question, candidates, top_n=top_n)
    print(f"-- Winning child chunks (top {len(children)}, reranked) --")
    for i, row in enumerate(children, start=1):
        text = row["rowJSON"]["text"]
        preview = text.replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        pages = page_numbers_for_expanded_chunk(row)
        page_range = f"{pages[0]}-{pages[-1]}" if pages else "?"
        print(f"  {i}. score={row['rerank_score']:.3f}  [{source} p.{page_range}]  "
              f"{len(text):>5} chars  {preview}...")

    expanded = expand_to_parent_chunks(children)
    print(f"\n-- Expanded (deduped) result: {len(children)} child chunk(s) -> "
          f"{len(expanded)} parent section(s) --")
    for i, row in enumerate(expanded, start=1):
        text = row["rowJSON"]["text"]
        preview = text.replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        title = row["rowJSON"].get("title", "?")
        pages = page_numbers_for_expanded_chunk(row)
        page_range = f"{pages[0]}-{pages[-1]}" if pages else "?"
        n_matched = len(row.get("matched_children", []))
        print(f"  {i}. score={row['rerank_score']:.3f}  [{source} p.{page_range}]  "
              f"{len(text):>6} chars  \"{title}\"  <- {n_matched} matched chunk(s)")
        print(f"     {preview}...")

    child_chars = sum(len(row["rowJSON"]["text"]) for row in children)
    expanded_chars = sum(len(row["rowJSON"]["text"]) for row in expanded)
    dedup_count = len(children) - len(expanded)
    print(f"\nDedup collapsed {dedup_count} chunk(s) sharing a parent with another winning chunk: "
          f"{dedup_count > 0}")
    print(f"Total context size: {child_chars} chars (children only) -> {expanded_chars} chars (expanded)")
    print("-" * 80)
    return children, expanded


example_results = {}   # question -> (children, expanded), filled in by the 3 examples below


**Cell #05**

## Three nutrition examples for the parent-chunk-expansion demonstration

Each chosen for a different reason a narrow child chunk struggles on its
own:

1. **A comparison question whose two halves live in the same section** —
   the running fiber example above. The winning child chunks for "soluble"
   and "insoluble" typically come from the same parent section, so
   expansion both fills in the missing half *and* dedups two chunks down
   to one expanded excerpt.
2. **A "what should be done" question whose single best chunk omits a
   caveat or a related list** — a narrow chunk about lab tests for a
   pediatric growth concern may not mention the full differential-diagnosis
   context sitting just above or below it in the same section.
3. **A full-pipeline question — hybrid + rerank + expand together** —
   showing that expansion composes with everything from the earlier
   notebooks rather than replacing it: hybrid picks candidates, rerank
   re-scores them, expansion swaps in context right before generation.


In [ ]:
EXAMPLE_QUESTIONS = [
    # 1. Comparison question -- both halves usually live in one section,
    #    so expansion both completes the answer and dedups two winners.
    "How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?",
    # 2. A single narrow chunk may omit the broader list/criteria around it.
    "What laboratory tests should be considered when evaluating a child for failure to thrive?",
    # 3. Full pipeline: hybrid + rerank + expand, on the familiar protein-RDA theme.
    "How many grams of protein per kilogram of body weight does the RDA recommend, "
    "and how does that differ for athletes?",
]


**Cell #07**

### Example 1 — a comparison question, both halves in one section


In [ ]:
example_results[EXAMPLE_QUESTIONS[0]] = show_expansion_comparison(EXAMPLE_QUESTIONS[0])


**Cell #09**

### Example 2 — a narrow chunk that may omit the broader criteria around it


In [ ]:
example_results[EXAMPLE_QUESTIONS[1]] = show_expansion_comparison(EXAMPLE_QUESTIONS[1])


**Cell #11**

### Example 3 — full pipeline setup (hybrid + rerank candidates, before expansion)

Reuses `hybrid_search()` from `stage2_ask_examples3_hybrid_search.ipynb` to
build the candidate pool, then reranks it — the same composition
`ask_question(use_hybrid=True, use_rerank=True, expand_to_parents=True)`
performs internally, shown here one step at a time.


In [ ]:
hybrid_candidates = hybrid_search(EXAMPLE_QUESTIONS[2], match_count=15)
example3_children = rerank_chunks(EXAMPLE_QUESTIONS[2], hybrid_candidates, top_n=NUM_CONTEXT_CHUNKS)
example3_expanded = expand_to_parent_chunks(example3_children)
example_results[EXAMPLE_QUESTIONS[2]] = (example3_children, example3_expanded)

print(f"Q: {EXAMPLE_QUESTIONS[2]}\n")
print(f"Hybrid+rerank winning child chunks: {len(example3_children)}")
print(f"After expand_to_parent_chunks(): {len(example3_expanded)} distinct parent section(s)")
for i, row in enumerate(example3_expanded, start=1):
    title = row["rowJSON"].get("title", "?")
    n_matched = len(row.get("matched_children", []))
    print(f"  {i}. \"{title}\" <- {n_matched} matched chunk(s), "
          f"dense_rank={row.get('dense_rank')} keyword_rank={row.get('keyword_rank')}")


**Cell #13**

## `ask_question(..., expand_to_parents=...)` end to end

`expand_to_parents` is an optional keyword argument on `ask_question` — it
defaults to `False`, so every existing call in
`stage2_ask_examples1/2/3.ipynb` behaves exactly as before. Passing
`expand_to_parents=True` runs expansion **last**, on whatever
`use_hybrid`/`use_rerank` already selected — it's the final step before
`build_expanded_context_block()` renders what Claude sees.


In [ ]:
demo_question = EXAMPLE_QUESTIONS[0]

baseline = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_rerank=True)
expanded_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS,
                                use_rerank=True, expand_to_parents=True)
full_pipeline_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS,
                                     use_hybrid=True, use_rerank=True, expand_to_parents=True)

print("Q:", demo_question)

print("\n-- ask_question(..., use_rerank=True) [baseline, no expansion] --")
print("chunks used:", baseline["chunks_used"], "| source pages:", baseline["source_pages"])
if baseline["short_answer"]:
    print("Short answer:", baseline["short_answer"])
print(baseline["answer"])

print("\n-- ask_question(..., use_rerank=True, expand_to_parents=True) --")
print("chunks used:", expanded_answer["chunks_used"], "| source pages:", expanded_answer["source_pages"])
print("used_parent_expansion:", expanded_answer["used_parent_expansion"])
if expanded_answer["short_answer"]:
    print("Short answer:", expanded_answer["short_answer"])
print(expanded_answer["answer"])

print("\n-- ask_question(..., use_hybrid=True, use_rerank=True, expand_to_parents=True) [full pipeline] --")
print("chunks used:", full_pipeline_answer["chunks_used"],
      "| candidates considered:", full_pipeline_answer["candidates_considered"])
print("source pages:", full_pipeline_answer["source_pages"])
if full_pipeline_answer["short_answer"]:
    print("Short answer:", full_pipeline_answer["short_answer"])
print(full_pipeline_answer["answer"])


**Cell #15**

## `expand_to_parent_chunks()` in isolation — the dedup rule, with toy data

`expand_to_parent_chunks()` is a small, focused function — it doesn't know
anything about *how* its input chunks were ranked, just "a list of rows
that each carry a `rowParentGUID`." Worth seeing on its own, with hand-built
toy rows sharing the module's own row shape, so the dedup rule (and which
ranking fields survive it) is visible without any network call.


In [ ]:
toy_children = [
    {"rowGUID": "child-soluble", "rowParentGUID": "parent-fiber-section", "rowOwnerGUID": "owner-1",
     "rerank_score": 0.94, "rowJSON": {"text": "Soluble fiber binds bile acids...", "source_key": "toy"}},
    {"rowGUID": "child-insoluble", "rowParentGUID": "parent-fiber-section", "rowOwnerGUID": "owner-1",
     "rerank_score": 0.81, "rowJSON": {"text": "Insoluble fiber adds bulk to stool...", "source_key": "toy"}},
    {"rowGUID": "child-unrelated", "rowParentGUID": "parent-other-section", "rowOwnerGUID": "owner-1",
     "rerank_score": 0.55, "rowJSON": {"text": "An unrelated chunk from a different section.", "source_key": "toy"}},
]

# Faked parent rows, keyed the way crud_chunks_parent.read_parent_row() would
# return them -- for this isolated demo we monkeypatch the lookup instead of
# hitting Supabase, so the dedup rule is visible without any network call.
import reusable_code.parent_chunk_expansion as pce

toy_parents = {
    "parent-fiber-section": {
        "rowGUID": "parent-fiber-section",
        "rowJSON": {
            "title": "Dietary Fiber: Soluble vs. Insoluble",
            "source_key": "toy",
            "text": "Full section: soluble fiber binds bile acids and lowers LDL; "
                    "insoluble fiber does not bind bile acids, adds bulk to stool, "
                    "and speeds transit time.",
        },
    },
    "parent-other-section": {
        "rowGUID": "parent-other-section",
        "rowJSON": {"title": "Unrelated Topic", "source_key": "toy", "text": "A different section entirely."},
    },
}
_original_read_parent_row = pce.read_parent_row
pce.read_parent_row = lambda guid, clients=None: toy_parents.get(guid)

toy_expanded = pce.expand_to_parent_chunks(toy_children)

pce.read_parent_row = _original_read_parent_row  # restore the real lookup

print(f"{'rowGUID':<20}{'matched_children':<45}rerank_score")
for row in toy_expanded:
    matched = [r['rowGUID'] for r in row['matched_children']]
    print(f"{row['rowGUID']:<20}{str(matched):<45}{row['rerank_score']}")

print(f"\n3 toy child chunks -> {len(toy_expanded)} expanded rows (2 shared a parent, 1 didn't):",
      len(toy_expanded) == 2)
print("The surviving row for the shared parent keeps the BEST child's rerank_score (0.94, not 0.81):",
      toy_expanded[0]["rerank_score"] == 0.94)


**Cell #17**

## Summary across the 3 examples

Same idea as the hybrid-search notebook's summary table: a quick scan of
how many winning child chunks each question retrieved, how many distinct
parent sections they collapsed into, and how much the context grew.


In [ ]:
print(f"{'#':<3} {'children':<10} {'parents':<9} {'dedup?':<8} {'chars before':<14} {'chars after':<14} question")
for i, question in enumerate(EXAMPLE_QUESTIONS, start=1):
    children, expanded = example_results[question]
    chars_before = sum(len(row["rowJSON"]["text"]) for row in children)
    chars_after = sum(len(row["rowJSON"]["text"]) for row in expanded)
    deduped = len(children) > len(expanded)
    print(f"{i:<3} {len(children):<10} {len(expanded):<9} {str(deduped):<8} "
          f"{chars_before:<14} {chars_after:<14} {question}")


**Cell #19**

## For stakeholders — what this means in plain terms

| Without expansion | With expansion |
| --- | --- |
| Claude only sees the 2–3 sentences that matched the question. | Claude sees the full section those sentences came from. |
| A comparison question ("X vs. Y") can get a one-sided answer if only one side's chunk won retrieval. | Both sides usually surface together, since they typically live in the same section. |
| A duplicate answer chunk-by-chunk if several winning chunks share a section. | One expanded excerpt per section — no repeated text, no wasted context budget. |
| No extra cost: zero additional API calls, since it only reads a row already sitting in Supabase. | Same: `expand_to_parent_chunks()` adds one lightweight database read per **distinct** parent, not per chunk. |

**Net effect:** more complete, better-grounded answers on comparison and
multi-part questions, at the cost of somewhat longer context per answer
(bounded by `max_parent_chars`, default 6000 characters per parent section)
— a good trade for a clinical nutrition Q&A system, where an incomplete
comparison is worse than a slightly longer one.

## For AI engineers — what this means technically

- **Zero schema change.** `rag11_chunks_parent_table` and the child table's
  `rowParentGUID` foreign key already existed for the parent/child chunking
  hierarchy — expansion is a pure query-time Python step
  (`reusable_code/parent_chunk_expansion.py`) on top of rows any retrieval
  method already returns.
- **Composable, not a replacement.** It runs *after* `retrieve_chunks()` /
  `hybrid_search()` / `rerank_chunks()` and *before*
  `build_expanded_context_block()` — see `ask_question(use_hybrid=...,
  use_rerank=..., expand_to_parents=...)` in Cell #14 above.
- **Dedup is "first occurrence wins."** The expanded row for a shared
  parent keeps the *best-ranked* input chunk's metadata
  (`rerank_score`/`rrf_score`/`cosine_distance`/...) — see the toy example
  in Cell #16, and `reciprocal_rank_fusion()` in `hybrid_search.py`, which
  uses the exact same rule when several ranked lists share a row.
- **Graceful degradation.** If a parent row is ever missing (shouldn't
  happen — it's a foreign key), the original child chunk is kept as-is
  rather than silently dropped — see `expanded_from_parent` on each row.
- **Bounded, not unbounded.** A parent chunk is a whole book *section*, not
  token-budgeted the way a child chunk is, so `max_parent_chars` (default
  `DEFAULT_MAX_PARENT_CHARS = 6000`) truncates an unusually long one; pass
  `max_parent_chars=None` to disable truncation.


**Cell #20**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock
recovery and conflict resolution), the same helper
`stage2_ask_examples3_hybrid_search.ipynb` uses -- shared via
`reusable_code.save_to_github`.


In [ ]:
from reusable_code import save_to_github

save_to_github("stage2_ask_examples4_parent_chunk_expansion.ipynb - parent-chunk expansion examples added")
